# Notebook 02: Preprocessing

This notebook merges the data, parses datetime, handles missing values, and shifts the target variable for forecasting.

In [1]:
import pandas as pd
import glob
import os
import warnings
warnings.filterwarnings('ignore')

## 1. Load the 3 Stations Data

In [2]:
files = glob.glob('../data/raw/**/*.csv', recursive=True)
target_stations = ['Aotizhongxin', 'Changping', 'Dingling']

dfs = []
for f in files:
    if any(station in f for station in target_stations):
        df = pd.read_csv(f)
        dfs.append(df)

df_raw = pd.concat(dfs, ignore_index=True)
print(f"Loaded {len(df_raw)} rows.")

Loaded 105192 rows.


## 2. Datetime Parsing

In [3]:
df_raw['datetime'] = pd.to_datetime(df_raw[['year', 'month', 'day', 'hour']])
df_raw.set_index('datetime', inplace=True)
df_raw.drop(columns=['No'], inplace=True, errors='ignore')

## 3. Missing Value Imputation
Since it's time series, we'll use linear interpolation for small gaps and forward fill for the rest.

In [4]:
def impute_missing(group):
    # Interpolate up to 3 consecutive missing hours
    group = group.interpolate(method='linear', limit=3)
    # Forward fill the rest, then backward fill if start is missing
    group = group.ffill().bfill()
    return group

numeric_cols = df_raw.select_dtypes(include=['float64', 'int64']).columns
df_raw[numeric_cols] = df_raw.groupby('station')[numeric_cols].apply(impute_missing).reset_index(level=0, drop=True)

# Wind direction (categorical) just forward fill
df_raw['wd'] = df_raw.groupby('station')['wd'].ffill().bfill()

## 4. Target Variable Shifting ($t+1$)
We are forecasting the *next hour's* PM2.5. We shift the PM2.5 column by -1 so that the row for 10:00 AM contains the PM2.5 value for 11:00 AM.

In [5]:
df_raw['target_PM2.5_t_plus_1'] = df_raw.groupby('station')['PM2.5'].shift(-1)

# Drop the very last row of each station because it has no future PM2.5
df_raw.dropna(subset=['target_PM2.5_t_plus_1'], inplace=True)

print("Final shape after shifting:", df_raw.shape)

Final shape after shifting: (105189, 18)


## 5. Save Processed Data

In [6]:
df_raw.to_csv('../data/processed/02_processed.csv')

OSError: Cannot save file into a non-existent directory: '../data/processed'